In [ ]:
# 1) Setup & Imports
!pip -q install torch torchvision torchaudio pandas numpy scikit-learn matplotlib

import os
import json
import math
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# 2) Configuration & Reproducibility
POS_PATH = "/content/trashed_crypto_dataset.csv"  # очищенный позитивный класс
NEG_PATH = "/content/non_crypto_tweets.csv"
OUT_DIR = "/content/models/custom-cnn-crypto"

MAX_LEN = 192
BATCH_SIZE = 48 
EMBED_DIM = 384
FILTER_SIZES = [2, 3, 4, 5, 6]
NUM_FILTERS = 192
DROPOUT = 0.35
EPOCHS = 16
LEARNING_RATE = 7e-5
WEIGHT_DECAY = 0.03
WARMUP_RATIO = 0.12
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.05
MIN_FREQ = 1
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [ ]:
# 3) Load & Clean CSV Data
SEP_POS = None  
SEP_NEG = ";"

POS_CAND_COLS = ["text", "tweet_text", "clean_text", "full_text"]
NEG_COL = "full_text"


def pick_pos_col(df: pd.DataFrame):
    for c in POS_CAND_COLS:
        if c in df.columns:
            return c
    raise ValueError("No text column found in positive dataset")


def clean_text_basic(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip()
    return s


def load_datasets(pos_path: str, neg_path: str):
    try:
        pos_raw = pd.read_csv(pos_path, sep=SEP_POS, engine="python")
    except Exception:
        pos_raw = pd.read_csv(pos_path, sep=";")
    neg_raw = pd.read_csv(neg_path, sep=SEP_NEG)

    pos_col = pick_pos_col(pos_raw)
    if NEG_COL not in neg_raw.columns:
        raise ValueError(f"Column '{NEG_COL}' missing in negative dataset")

    pos_df = pos_raw[[pos_col]].rename(columns={pos_col: "text"})
    neg_df = neg_raw[[NEG_COL]].rename(columns={NEG_COL: "text"})

    pos_df["text"] = pos_df["text"].apply(clean_text_basic)
    neg_df["text"] = neg_df["text"].apply(clean_text_basic)

    pos_df = pos_df[pos_df["text"].ne("")].copy()
    neg_df = neg_df[neg_df["text"].ne("")].copy()

    pos_df["label"] = 1
    neg_df["label"] = 0
    return pos_df, neg_df

pos_df, neg_df = load_datasets(POS_PATH, NEG_PATH)
print(pos_df.shape, neg_df.shape)
print(pos_df.head(2))
print(neg_df.head(2))


In [ ]:
# 4) Train/Val/Test Split with Class Balance (maximize available data)
from sklearn.model_selection import train_test_split

def make_balanced_split(pos_df, neg_df):
    min_len = min(len(pos_df), len(neg_df))
    test_size = max(int(0.1 * min_len), 1)
    val_size = max(int(0.1 * min_len), 1)
    train_size = max(min_len - test_size - val_size, 1)

    pos_sample = pos_df.sample(min_len, random_state=SEED).reset_index(drop=True)
    neg_sample = neg_df.sample(min_len, random_state=SEED).reset_index(drop=True)

    pos_train, pos_temp = train_test_split(pos_sample, test_size=test_size + val_size, random_state=SEED, shuffle=True)
    pos_val, pos_test = train_test_split(pos_temp, test_size=test_size, random_state=SEED, shuffle=True)

    neg_train, neg_temp = train_test_split(neg_sample, test_size=test_size + val_size, random_state=SEED, shuffle=True)
    neg_val, neg_test = train_test_split(neg_temp, test_size=test_size, random_state=SEED, shuffle=True)

    train_df = pd.concat([pos_train, neg_train], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_df = pd.concat([pos_val, neg_val], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_df = pd.concat([pos_test, neg_test], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

    print(f"Train size: {len(train_df)} (pos {len(pos_train)}, neg {len(neg_train)})")
    print(f"Val size: {len(val_df)} (pos {len(pos_val)}, neg {len(neg_val)})")
    print(f"Test size: {len(test_df)} (pos {len(pos_test)}, neg {len(neg_test)})")
    return train_df, val_df, test_df

train_df, val_df, test_df = make_balanced_split(pos_df, neg_df)
print(train_df.label.value_counts())
print(val_df.label.value_counts())
print(test_df.label.value_counts())


In [ ]:
# 5) Build Tokenizer & Vocabulary (whitespace/regex)
TOKEN_PATTERN = re.compile(r"\b\w+\b")
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"


def tokenize(text: str):
    return TOKEN_PATTERN.findall(text.lower())


def build_vocab(texts, min_freq=MIN_FREQ):
    freq = {}
    for t in texts:
        for tok in tokenize(t):
            freq[tok] = freq.get(tok, 0) + 1
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for tok, c in sorted(freq.items(), key=lambda x: (-x[1], x[0])):
        if c >= min_freq:
            vocab[tok] = len(vocab)
    inv_vocab = {i: t for t, i in vocab.items()}
    return vocab, inv_vocab

all_texts = pd.concat([train_df["text"], val_df["text"], test_df["text"]], ignore_index=True)
vocab, inv_vocab = build_vocab(all_texts.tolist(), min_freq=MIN_FREQ)
print("Vocab size:", len(vocab))

os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)


In [ ]:
# 6) Text to Tensor Dataset & DataLoaders

def encode(text: str, vocab, max_len=MAX_LEN):
    toks = tokenize(text)
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in toks][:max_len]
    if len(ids) < max_len:
        ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

class TextDataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids = encode(row.text, self.vocab, MAX_LEN)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(row.label, dtype=torch.long)

train_dataset = TextDataset(train_df, vocab)
val_dataset = TextDataset(val_df, vocab)
test_dataset = TextDataset(test_df, vocab)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print("Batches train/val/test:", len(train_loader), len(val_loader), len(test_loader))


In [ ]:
# 7) Define Architectures (baseline + stronger variants)
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes, num_classes=2, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):  
        emb = self.embedding(x)             
        emb = emb.transpose(1, 2)            
        conv_outs = [torch.relu(conv(emb)) for conv in self.convs]
        pooled = [torch.max(c, dim=2).values for c in conv_outs]
        cat = torch.cat(pooled, dim=1)      
        cat = self.dropout(cat)
        logits = self.fc(cat)
        return logits


class BiLSTMMaxPool(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim=256, num_layers=1, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)  # [B, L, E]
        out, _ = self.lstm(emb)  # [B, L, 2H]
        pooled = torch.max(out, dim=1).values
        pooled = self.dropout(pooled)
        return self.fc(pooled)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):  # x: [B, L, E]
        x = x + self.pe[:, : x.size(1), :]
        return x


class TransformerEncoderSmall(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads=8,
        ff_dim=512,
        num_layers=2,
        num_classes=2,
        dropout=0.2,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoding = PositionalEncoding(embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)  # [B, L, E]
        emb = self.pos_encoding(emb)
        enc = self.encoder(emb)
        pooled = torch.max(enc, dim=1).values
        pooled = self.dropout(pooled)
        return self.fc(pooled)


print("Defined models: TextCNN, BiLSTMMaxPool, TransformerEncoderSmall")


In [ ]:
# 8) Train three models sequentially on the same data
from copy import deepcopy

label_counts = train_df.label.value_counts().to_dict()
cls_weights = torch.tensor([
    (label_counts.get(0, 1.0)),
    (label_counts.get(1, 1.0)),
], dtype=torch.float)
cls_weights = (cls_weights.sum() / (2.0 * cls_weights)).to(device)


def make_criterion():
    return nn.CrossEntropyLoss(weight=cls_weights, label_smoothing=LABEL_SMOOTHING)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            all_labels.append(y.cpu())
            all_probs.append(probs.cpu())
            all_preds.append(preds.cpu())
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()
    all_preds = torch.cat(all_preds).numpy()

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    roc_auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan")
    pr_auc = average_precision_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan")
    cm = confusion_matrix(all_labels, all_preds)

    avg_loss = total_loss / len(loader.dataset)
    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "cm": cm,
        "labels": all_labels,
        "probs": all_probs,
        "preds": all_preds,
    }


def train_model(model_name, build_fn, lr=LEARNING_RATE, epochs=EPOCHS):
    criterion = make_criterion()
    model = build_fn().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, epochs * len(train_loader))
    warmup_steps = int(WARMUP_RATIO * total_steps)

    def lr_lambda(current_step: int):
        if current_step < warmup_steps:
            return float(current_step + 1) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": [],
        "val_roc_auc": [],
        "val_pr_auc": [],
        "lr": [],
    }
    best_f1 = -1.0
    best_state = deepcopy(model.state_dict())
    patience_counter = 0
    patience = 4

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            if GRAD_CLIP:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item() * y.size(0)
            history["lr"].append(optimizer.param_groups[0]["lr"])

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["acc"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])
        history["val_f1"].append(val_metrics["f1"])
        history["val_roc_auc"].append(val_metrics["roc_auc"])
        history["val_pr_auc"].append(val_metrics["pr_auc"])

        improved = val_metrics["f1"] > best_f1
        if improved:
            best_f1 = val_metrics["f1"]
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f"[{model_name}] Epoch {epoch}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"acc={val_metrics['acc']:.4f} | "
            f"prec={val_metrics['precision']:.4f} | "
            f"recall={val_metrics['recall']:.4f} | "
            f"f1={val_metrics['f1']:.4f} | "
            f"roc_auc={val_metrics['roc_auc']:.4f} | "
            f"pr_auc={val_metrics['pr_auc']:.4f}"
        )

        if patience_counter >= patience:
            print(f"[{model_name}] Early stopping triggered")
            break

    # Load best state by val F1
    model.load_state_dict(best_state)
    final_eval = evaluate(model, test_loader, criterion)
    return model, history, final_eval


models_to_run = [
    {
        "name": "TextCNN",
        "builder": lambda: TextCNN(
            vocab_size=len(vocab),
            embed_dim=EMBED_DIM,
            num_filters=NUM_FILTERS,
            filter_sizes=FILTER_SIZES,
            num_classes=2,
            dropout=DROPOUT,
        ),
        "lr": LEARNING_RATE,
        "epochs": EPOCHS,
    },
    {
        "name": "BiLSTMMaxPool",
        "builder": lambda: BiLSTMMaxPool(
            vocab_size=len(vocab),
            embed_dim=EMBED_DIM,
            hidden_dim=256,
            num_layers=1,
            num_classes=2,
            dropout=0.3,
        ),
        "lr": 1e-4,
        "epochs": EPOCHS,
    },
    {
        "name": "TransformerEncoderSmall",
        "builder": lambda: TransformerEncoderSmall(
            vocab_size=len(vocab),
            embed_dim=EMBED_DIM,
            num_heads=8,
            ff_dim=512,
            num_layers=2,
            num_classes=2,
            dropout=0.2,
        ),
        "lr": 8e-5,
        "epochs": EPOCHS,
    },
]

trained_models = {}
all_results = {}
histories = {}

for cfg in models_to_run:
    torch.cuda.empty_cache()
    model, history, final_eval = train_model(cfg["name"], cfg["builder"], lr=cfg["lr"], epochs=cfg["epochs"])
    trained_models[cfg["name"]] = model
    histories[cfg["name"]] = history
    all_results[cfg["name"]] = final_eval
    print(f"\nFinal test metrics for {cfg['name']}:")
    for k, v in final_eval.items():
        if k in {"labels", "probs", "preds", "cm"}:
            continue
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

print("\nCompleted training of all models.")


In [ ]:
# 9) Detailed Test Reports per model
from sklearn.metrics import classification_report

for model_name, res in all_results.items():
    print(f"\n=== {model_name} ===")
    report = classification_report(
        res["labels"], res["preds"], target_names=["non-crypto", "crypto"], digits=4, zero_division=0
    )
    cm = res["cm"]
    print("Classification report:\n", report)
    print("Confusion matrix:\n", cm)
    cm_norm = cm.astype(np.float32) / cm.sum(axis=1, keepdims=True)
    print("Normalized confusion matrix (per true class):\n", np.round(cm_norm, 4))


In [ ]:
# 10) Plots: Loss curves and AUCs per model
from sklearn.metrics import roc_curve, precision_recall_curve

for model_name, history in histories.items():
    plt.figure(figsize=(12, 4))
    plt.suptitle(model_name)

    plt.subplot(1, 3, 1)
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Loss")

    plt.subplot(1, 3, 2)
    plt.plot(history["val_f1"], label="Val F1")
    plt.plot(history["val_acc"], label="Val Acc")
    plt.plot(history["val_precision"], label="Val Prec")
    plt.plot(history["val_recall"], label="Val Recall")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend()
    plt.title("Val Metrics")

    plt.subplot(1, 3, 3)
    plt.plot(history["val_roc_auc"], label="Val ROC-AUC")
    plt.plot(history["val_pr_auc"], label="Val PR-AUC")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend()
    plt.title("Val AUCs")

    plt.tight_layout()
    plt.show()

# ROC/PR curves on final models (test)
for model_name, res in all_results.items():
    if len(np.unique(res["labels"])) > 1:
        fpr, tpr, _ = roc_curve(res["labels"], res["probs"])
        prec_curve, rec_curve, _ = precision_recall_curve(res["labels"], res["probs"])

        plt.figure(figsize=(10, 4))
        plt.suptitle(model_name + " (test)")

        plt.subplot(1, 2, 1)
        plt.plot(fpr, tpr, label=f"ROC-AUC={res['roc_auc']:.4f}")
        plt.plot([0, 1], [0, 1], "--", color="gray")
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.title("ROC Curve")
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(rec_curve, prec_curve, label=f"PR-AUC={res['pr_auc']:.4f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("PR Curve")
        plt.legend()

        plt.tight_layout()
        plt.show()


In [ ]:
# 11) Save artifacts per model (weights, metrics, history, curves)
os.makedirs(OUT_DIR, exist_ok=True)

for model_name, res in all_results.items():
    model_dir = os.path.join(OUT_DIR, model_name.lower())
    os.makedirs(model_dir, exist_ok=True)

    metrics_to_save = {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in res.items() if k not in {"labels", "probs", "preds", "cm"}}
    metrics_to_save["confusion_matrix"] = res["cm"].tolist()
    metrics_to_save["history"] = histories.get(model_name, {})

    config = {
        "POS_PATH": POS_PATH,
        "NEG_PATH": NEG_PATH,
        "OUT_DIR": model_dir,
        "MAX_LEN": MAX_LEN,
        "BATCH_SIZE": BATCH_SIZE,
        "EMBED_DIM": EMBED_DIM,
        "FILTER_SIZES": FILTER_SIZES,
        "NUM_FILTERS": NUM_FILTERS,
        "DROPOUT": DROPOUT,
        "EPOCHS": EPOCHS,
        "LEARNING_RATE": LEARNING_RATE,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "WARMUP_RATIO": WARMUP_RATIO,
        "GRAD_CLIP": GRAD_CLIP,
        "LABEL_SMOOTHING": LABEL_SMOOTHING,
        "MIN_FREQ": MIN_FREQ,
        "SEED": SEED,
    }

    # Save model weights
    model_path = os.path.join(model_dir, "model.pt")
    torch.save(trained_models[model_name].state_dict(), model_path)

    # Save config and metrics
    with open(os.path.join(model_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    with open(os.path.join(model_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics_to_save, f, ensure_ascii=False, indent=2)

    # Save vocab (shared)
    with open(os.path.join(model_dir, "vocab.json"), "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False)

    # Save curves data (test)
    if len(np.unique(res["labels"])) > 1:
        fpr, tpr, roc_thr = roc_curve(res["labels"], res["probs"])
        prec_curve, rec_curve, pr_thr = precision_recall_curve(res["labels"], res["probs"])
        np.save(os.path.join(model_dir, "roc_curve.npy"), np.vstack([fpr, tpr, roc_thr]))
        np.save(os.path.join(model_dir, "pr_curve.npy"), np.vstack([prec_curve, rec_curve, pr_thr]))

print("Saved models and artifacts to", OUT_DIR)


In [ ]:
# 4.1) Save balanced splits for reuse (intermediate artifact)
os.makedirs(OUT_DIR, exist_ok=True)
train_split_path = os.path.join(OUT_DIR, "train_split.csv")
val_split_path = os.path.join(OUT_DIR, "val_split.csv")
test_split_path = os.path.join(OUT_DIR, "test_split.csv")
train_df.to_csv(train_split_path, index=False)
val_df.to_csv(val_split_path, index=False)
test_df.to_csv(test_split_path, index=False)
print("Saved splits:", train_split_path, val_split_path, test_split_path)
